# 02 — Generación del Dataset Artificial

**Objetivo:** Transformar los parámetros reales extraídos del EDA (notebook 01) en un
dataset artificial alineado al esquema `Draft`, listo para entrenar el modelo Isolation Forest.

**Prerequisito:** ejecutar `01_data_extraction_and_eda.ipynb` para generar `datasets/eda_params.json`.

**Output:** `datasets/properties_synthetic_v1.parquet`

In [1]:
import json
import pathlib
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

PARAMS_PATH = pathlib.Path('../datasets/eda_params.json')
OUT_PATH    = pathlib.Path('../datasets/properties_synthetic_v1.parquet')
N_NORMAL    = 6_000  # registros normales
ANOMALY_PCT = 0.04   # 4% de anomalías para validación

params = json.loads(PARAMS_PATH.read_text())
print(f'Tipos de propiedad disponibles: {list(params.keys())}')

Tipos de propiedad disponibles: ['House', 'Apartment', 'Store']


## 1. Generación de registros normales

Muestreamos con distribuciones calibradas desde el EDA. Los campos ausentes en
Properati (`bathrooms`, `parkingSpaces`, `constructionYear`, `amenityIds`, `condominium`)
se imputan con distribuciones razonables para el mercado MX.

In [2]:
def clamp(arr, lo, hi):
    return np.clip(arr, lo, hi)

def sample_normal(n, mean, std, lo, hi):
    """Muestrea distribución normal truncada en [lo, hi]."""
    vals = rng.normal(mean, std, n * 3)  # sobregenerar
    vals = vals[(vals >= lo) & (vals <= hi)]
    if len(vals) < n:
        extra = rng.uniform(lo, hi, n - len(vals))
        vals = np.concatenate([vals, extra])
    return vals[:n]

# Mapeo Properati → tipos de producción (en BD: property_type.name)
PT_MAP = {
    'House'     : 'Casa',
    'Apartment' : 'Departamento',
    'Store'     : 'Local Comercial',
}

# Parámetros manuales para tipos sin cobertura en Properati MX
EXTRA_PARAMS = {
    'Terreno': {
        'area_mean': 350, 'area_std': 250, 'area_min': 50,  'area_max': 5000,
        'price_per_m2_mean': 3500,  'price_per_m2_std': 2500,
        'price_per_m2_min': 300,    'price_per_m2_max': 25000,
        'bedrooms_median': 0, 'bedrooms_std': 0,
        'n_samples': 400,
    },
    'Oficina': {
        'area_mean': 90,  'area_std': 70,  'area_min': 15,  'area_max': 600,
        'price_per_m2_mean': 18000, 'price_per_m2_std': 8000,
        'price_per_m2_min': 5000,   'price_per_m2_max': 60000,
        'bedrooms_median': 0, 'bedrooms_std': 0,
        'n_samples': 400,
    },
    'Bodega': {
        'area_mean': 500, 'area_std': 350, 'area_min': 50,  'area_max': 4000,
        'price_per_m2_mean': 7000,  'price_per_m2_std': 4000,
        'price_per_m2_min': 800,    'price_per_m2_max': 25000,
        'bedrooms_median': 0, 'bedrooms_std': 0,
        'n_samples': 300,
    },
}

# Distribuciones por tipo de producción para campos ausentes en Properati
BATH_BY_TYPE = {
    'Casa'           : (2.0, 1.0),
    'Departamento'   : (1.5, 0.7),
    'Terreno'        : (0.0, 0.0),
    'Local Comercial': (1.0, 0.5),
    'Oficina'        : (1.5, 0.5),
    'Bodega'         : (1.0, 0.5),
}
PARKING_BY_TYPE = {
    'Casa'           : (1.5, 1.0),
    'Departamento'   : (1.0, 0.7),
    'Terreno'        : (0.0, 0.0),
    'Local Comercial': (1.0, 0.8),
    'Oficina'        : (2.0, 1.5),
    'Bodega'         : (2.0, 1.5),
}
CONDO_BY_TYPE = {
    'Casa'           : 0.30,
    'Departamento'   : 0.80,
    'Terreno'        : 0.05,
    'Local Comercial': 0.15,
    'Oficina'        : 0.50,
    'Bodega'         : 0.10,
}
YEAR_BY_TYPE = {
    'Casa'           : (1980, 15),
    'Departamento'   : (1995, 12),
    'Terreno'        : (1970, 20),
    'Local Comercial': (1990, 12),
    'Oficina'        : (2000, 10),
    'Bodega'         : (1990, 12),
}

def _defaults(pt):
    """Retorna (bath_mean, bath_std), (park_mean, park_std), p_condo, (year_mean, year_std)."""
    return (
        BATH_BY_TYPE.get(pt, (1.0, 0.5)),
        PARKING_BY_TYPE.get(pt, (1.0, 0.7)),
        CONDO_BY_TYPE.get(pt, 0.2),
        YEAR_BY_TYPE.get(pt, (1990, 15)),
    )

In [3]:
# Remap Properati names → nombres de producción, y agregar tipos extra
params_prod = {PT_MAP.get(k, k): v for k, v in params.items()}
params_prod.update(EXTRA_PARAMS)

types   = list(params_prod.keys())
weights = np.array([params_prod[t]['n_samples'] for t in types], dtype=float)
weights /= weights.sum()
counts  = (weights * N_NORMAL).astype(int)
counts[-1] += N_NORMAL - counts.sum()  # ajustar redondeo

print('Distribución de registros por tipo:')
for t, n in zip(types, counts):
    print(f'  {t:<20}: {n:,}')

rows = []
for pt, n in zip(types, counts):
    p = params_prod[pt]
    bath_cfg, park_cfg, p_condo, year_cfg = _defaults(pt)

    area  = sample_normal(n, p['area_mean'], p['area_std'], p['area_min'], p['area_max'])
    ppm2  = sample_normal(n, p['price_per_m2_mean'], p['price_per_m2_std'],
                          p['price_per_m2_min'], p['price_per_m2_max'])
    price = area * ppm2  # precio consistente con área y precio/m²
    beds  = np.round(clamp(rng.normal(p['bedrooms_median'], max(p['bedrooms_std'], 0.5), n),
                           0, 10)).astype(int)

    bath_m, bath_s = bath_cfg
    baths = np.round(clamp(rng.normal(bath_m, bath_s, n), 0, 8), 1)

    park_m, park_s = park_cfg
    parking = np.round(clamp(rng.normal(park_m, park_s, n), 0, 10)).astype(int)

    year_m, year_s = year_cfg
    year = np.round(clamp(rng.normal(year_m, year_s, n), 1900, 2024)).astype(int)

    condo    = rng.random(n) < p_condo
    n_amen   = rng.integers(0, 15, n)
    n_images = rng.integers(1, 25, n)

    for i in range(n):
        rows.append({
            'propertyType'    : pt,
            'areaM2'          : round(area[i], 1),
            'listedPrice'     : round(price[i], 2),
            'pricePerM2'      : round(ppm2[i], 2),
            'bedrooms'        : int(beds[i]),
            'bathrooms'       : float(baths[i]),
            'parkingSpaces'   : int(parking[i]),
            'constructionYear': int(year[i]),
            'antiguedad'      : 2026 - int(year[i]),
            'condominium'     : int(condo[i]),
            'n_amenities'     : int(n_amen[i]),
            'total_images'    : int(n_images[i]),
            'is_anomaly'      : False,
            'anomaly_type'    : None,
        })

df_normal = pd.DataFrame(rows)
print(f'\nRegistros normales generados: {len(df_normal):,}')
print(f"Tipos en el dataset: {df_normal['propertyType'].unique().tolist()}")
df_normal.describe()

Distribución de registros por tipo:
  Casa                : 4,299
  Departamento        : 1,380
  Local Comercial     : 276
  Terreno             : 16
  Oficina             : 16
  Bodega              : 13

Registros normales generados: 6,000
Tipos en el dataset: ['Casa', 'Departamento', 'Local Comercial', 'Terreno', 'Oficina', 'Bodega']


,areaM2,listedPrice,pricePerM2,bedrooms,bathrooms,parkingSpaces,constructionYear,antiguedad,condominium,n_amenities,total_images
count,6000.000000,6.000000e+03,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000
mean,285.712150,3.625210e+06,13794.304818,2.693333,1.840750,1.385667,1983.479833,42.520167,0.410000,6.986167,12.604500
std,180.935149,2.960043e+06,8917.411375,1.292246,0.951729,0.969064,15.628019,15.628019,0.491874,4.326827,6.856207
min,10.100000,4.215340e+03,102.010000,0.000000,0.000000,0.000000,1923.000000,2.000000,0.000000,0.000000,1.000000
25%,136.275000,1.220653e+06,7408.892500,2.000000,1.200000,1.000000,1973.000000,32.000000,0.000000,3.000000,7.000000
50%,250.500000,2.896658e+06,12707.250000,3.000000,1.800000,1.000000,1984.000000,42.000000,0.000000,7.000000,13.000000
75%,434.225000,5.333492e+06,18286.050000,4.000000,2.500000,2.000000,1994.000000,53.000000,1.000000,11.000000,19.000000
max,1425.800000,1.468507e+07,52799.960000,10.000000,6.100000,6.000000,2024.000000,103.000000,1.000000,14.000000,24.000000


## 2. Inyección de anomalías sintéticas

Estas anomalías **solo sirven para validar** que el modelo las detecta; **no se incluyen
en el entrenamiento**. Se marcan con `is_anomaly=True` y un `anomaly_type` descriptivo.

In [4]:
N_ANOMALY = int(N_NORMAL * ANOMALY_PCT)
anomaly_rows = []

# Tomar una base aleatoria de registros normales para partir
base = df_normal.sample(N_ANOMALY, random_state=99).copy().reset_index(drop=True)

anomaly_types = [
    'precio_absurdo',       # listedPrice 10× el rango normal
    'area_imposible',       # areaM2 extremo (< 5 m² o > 8 000 m²)
    'ppm2_incoherente',     # pricePerM2 no coincide con price/area
    'ano_construccion_inv', # constructionYear > 2026 o < 1900
]

chunk = N_ANOMALY // len(anomaly_types)
extra = N_ANOMALY - chunk * len(anomaly_types)

start = 0
for i, atype in enumerate(anomaly_types):
    end = start + chunk + (1 if i < extra else 0)
    rows_a = base.iloc[start:end].copy()

    if atype == 'precio_absurdo':
        rows_a['listedPrice']  = rows_a['listedPrice'] * rng.uniform(8, 20, len(rows_a))
        rows_a['pricePerM2']   = rows_a['listedPrice'] / rows_a['areaM2']

    elif atype == 'area_imposible':
        # Mitad demasiado pequeña, mitad demasiado grande
        half = len(rows_a) // 2
        rows_a.iloc[:half, rows_a.columns.get_loc('areaM2')] = rng.uniform(1, 5, half)
        rows_a.iloc[half:, rows_a.columns.get_loc('areaM2')] = rng.uniform(8_000, 50_000, len(rows_a) - half)
        rows_a['pricePerM2'] = rows_a['listedPrice'] / rows_a['areaM2']

    elif atype == 'ppm2_incoherente':
        # pricePerM2 muy diferente al calculado de price/area
        rows_a['pricePerM2'] = rows_a['pricePerM2'] * rng.choice([0.05, 15], len(rows_a))

    elif atype == 'ano_construccion_inv':
        half = len(rows_a) // 2
        rows_a.iloc[:half, rows_a.columns.get_loc('constructionYear')] = rng.integers(2027, 2100, half)
        rows_a.iloc[half:, rows_a.columns.get_loc('constructionYear')] = rng.integers(1800, 1899, len(rows_a) - half)
        rows_a['antiguedad'] = 2026 - rows_a['constructionYear']

    rows_a['is_anomaly']  = True
    rows_a['anomaly_type'] = atype
    anomaly_rows.append(rows_a)
    start = end

df_anomaly = pd.concat(anomaly_rows, ignore_index=True)
print(f'Anomalías generadas: {len(df_anomaly):,}')
print(df_anomaly['anomaly_type'].value_counts())

Anomalías generadas: 240
anomaly_type
precio_absurdo          60
area_imposible          60
ppm2_incoherente        60
ano_construccion_inv    60
Name: count, dtype: int64


## 3. Combinar y guardar

In [5]:
df_final = pd.concat([df_normal, df_anomaly], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)  # mezclar

print(f'Dataset final: {len(df_final):,} filas')
print(f"Distribución is_anomaly:\n{df_final['is_anomaly'].value_counts()}")
print(f"\n% anomalías: {df_final['is_anomaly'].mean():.2%}")
df_final.head()

Dataset final: 6,240 filas
Distribución is_anomaly:
is_anomaly
False    6000
True      240
Name: count, dtype: int64

% anomalías: 3.85%


,propertyType,areaM2,listedPrice,pricePerM2,bedrooms,bathrooms,parkingSpaces,constructionYear,antiguedad,condominium,n_amenities,total_images,is_anomaly,anomaly_type
0,Casa,367.6,4933433.44,13421.19,2,2.6,2,1986,40,0,2,9,False,None
1,Casa,284.3,5920899.26,20824.68,2,2.2,2,1971,55,0,10,16,False,None
2,Departamento,234.4,6035620.22,25745.79,2,1.9,1,2007,19,1,7,7,False,None
3,Casa,544.4,8657651.69,15903.74,2,1.0,2,1977,49,0,3,14,False,None
4,Casa,249.9,3353187.52,13418.01,3,1.2,2,1975,51,0,1,19,False,None


In [6]:
# Verificar que no hay nulos en features principales
FEATURE_COLS = [
    'areaM2', 'listedPrice', 'pricePerM2', 'bedrooms', 'bathrooms',
    'parkingSpaces', 'constructionYear', 'antiguedad', 'condominium',
    'n_amenities', 'total_images',
]
null_count = df_final[FEATURE_COLS].isnull().sum()
assert null_count.sum() == 0, f'Nulos encontrados en features:\n{null_count[null_count > 0]}'
print('✅ Sin nulos en features principales')
print(f'✅ Columnas presentes: {FEATURE_COLS}')

✅ Sin nulos en features principales
✅ Columnas presentes: ['areaM2', 'listedPrice', 'pricePerM2', 'bedrooms', 'bathrooms', 'parkingSpaces', 'constructionYear', 'antiguedad', 'condominium', 'n_amenities', 'total_images']


In [7]:
df_final.to_parquet(OUT_PATH, index=False)
print(f'Dataset guardado en: {OUT_PATH.resolve()}')
print(f'Tamaño: {OUT_PATH.stat().st_size / 1024:.1f} KB')

# Verificación de lectura
df_check = pd.read_parquet(OUT_PATH)
print(f'\nVerificación — shape: {df_check.shape}')
print(f'Columnas: {df_check.columns.tolist()}')

Dataset guardado en: /home/aleosh/Documentos/Ingeniería en Software/9no Cuatrimestre/Integrador/vps/vivia-ai/datasets/properties_synthetic_v1.parquet
Tamaño: 164.8 KB

Verificación — shape: (6240, 14)
Columnas: ['propertyType', 'areaM2', 'listedPrice', 'pricePerM2', 'bedrooms', 'bathrooms', 'parkingSpaces', 'constructionYear', 'antiguedad', 'condominium', 'n_amenities', 'total_images', 'is_anomaly', 'anomaly_type']


## 4. Resumen y siguientes pasos

El dataset `properties_synthetic_v1.parquet` está listo para la fase de entrenamiento.

**Siguientes pasos:**
1. `src/anomaly_detector_api/training/features.py` — feature engineering compartido
2. `src/anomaly_detector_api/training/train_isolation_forest.py` — entrenamiento
3. Evaluar contra el bloque de anomalías inyectadas (`is_anomaly=True`)
4. Persistir modelo en `models_registry/anomaly/`